In [2]:
"""
Video Understanding and Prompt Generation Script (Updated)
==========================================================

This version fixes:
- PySceneDetect deprecation (uses open_video)
- Faster BLIP processor (`use_fast=True`)
- Automatic local caching for offline Hugging Face models
- Adjustable scene detection threshold
- Cleaner tqdm + fallback progress bar
"""

import os
import cv2
import torch
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

# ============================================================
# Configuration
# ============================================================
INPUT_PATH = r"C:/Users/JARE WORKS/Videos/Captures"
OUTPUT_DIR = "video_analysis_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SCENE_THRESHOLD = 20.0  # Lower = more scenes detected


# ============================================================
# 1. Scene Detection and Keyframe Extraction
# ============================================================
def extract_keyframes(video_path, output_dir):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_output_dir = os.path.join(output_dir, video_name)
    os.makedirs(video_output_dir, exist_ok=True)

    print(f"🔍 Detecting scenes for: {video_name}")

    video = open_video(video_path)
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=SCENE_THRESHOLD))

    scene_manager.detect_scenes(video)
    scene_list = scene_manager.get_scene_list()

    cap = cv2.VideoCapture(video_path)
    keyframes = []

    if not scene_list:
        print("⚠️  No scenes detected.")
        return []

    for i, (start, end) in enumerate(scene_list):
        frame_number = (start.get_frames() + end.get_frames()) // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        if ret:
            filename = os.path.join(video_output_dir, f"scene_{i:03d}.jpg")
            cv2.imwrite(filename, frame)
            keyframes.append(filename)

    cap.release()
    return keyframes


# ============================================================
# 2. Frame Captioning using BLIP
# ============================================================
def describe_frames(frame_paths):
    if not frame_paths:
        return []

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🧠 Using device: {device}")

    processor = BlipProcessor.from_pretrained(
        "Salesforce/blip-image-captioning-base", use_fast=True
    )
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    ).to(device)

    captions = []
    for frame in tqdm(frame_paths, desc="🖼️  Describing keyframes"):
        image = cv2.cvtColor(cv2.imread(frame), cv2.COLOR_BGR2RGB)
        inputs = processor(image, return_tensors="pt").to(device)
        out = model.generate(**inputs, max_length=50)
        caption = processor.decode(out[0], skip_special_tokens=True)
        captions.append(caption)

    return captions


# ============================================================
# 3. Video-Level Summarization
# ============================================================
def summarize_video(captions):
    if not captions:
        return "No visual descriptions generated."

    print("✍️  Summarizing video...")
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    combined_text = " ".join(captions)
    summary = summarizer(
        combined_text, max_length=200, min_length=60, do_sample=False
    )[0]["summary_text"]
    return summary


# ============================================================
# 4. Prompt Generation
# ============================================================
def generate_creation_prompt(summary):
    return (
        "Recreate a video based on the following detailed description:\n\n"
        f"{summary}\n\n"
        "Ensure realistic lighting, natural motion, and emotional continuity. "
        "Add creative camera angles or subtle context enhancements."
    )


# ============================================================
# 5. Full Pipeline
# ============================================================
def analyze_video(video_path, output_dir):
    print(f"\n=== Processing Video: {video_path} ===")
    keyframes = extract_keyframes(video_path, output_dir)

    if not keyframes:
        print("No keyframes generated, skipping this video.")
        return

    captions = describe_frames(keyframes)
    summary = summarize_video(captions)
    prompt = generate_creation_prompt(summary)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    text_output = os.path.join(output_dir, f"{video_name}_summary.txt")

    with open(text_output, "w", encoding="utf-8") as f:
        f.write("=== Video Summary ===\n")
        f.write(summary + "\n\n")
        f.write("=== Recreation Prompt ===\n")
        f.write(prompt)

    print(f"\n✅ Analysis completed for {video_name}")
    print(f"Results saved to: {text_output}")
    print("\n--- SUMMARY ---")
    print(summary)
    print("\n--- CREATION PROMPT ---")
    print(prompt)


# ============================================================
# 6. Entry Point
# ============================================================
if __name__ == "__main__":
    if os.path.isfile(INPUT_PATH):
        analyze_video(INPUT_PATH, OUTPUT_DIR)
    elif os.path.isdir(INPUT_PATH):
        video_files = [
            os.path.join(INPUT_PATH, f)
            for f in os.listdir(INPUT_PATH)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]
        for video in video_files:
            analyze_video(video, OUTPUT_DIR)
    else:
        print("❌ Invalid input path. Please provide a video file or folder.")



=== Processing Video: C:/Users/JARE WORKS/Videos/Captures\Discover Top Nigerian Software Solutions - Google Chrome 2025-05-06 21-38-57.mp4 ===
🔍 Detecting scenes for: Discover Top Nigerian Software Solutions - Google Chrome 2025-05-06 21-38-57
🧠 Using device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
🖼️  Describing keyframes: 100%|██████████| 2/2 [00:03<00:00,  1.67s/it]


✍️  Summarizing video...


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

Error while downloading from https://huggingface.co/Salesforce/blip-image-captioning-base/resolve/refs%2Fpr%2F52/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
